# Sentiment Analysis Notebook

This notebook demonstrates the full pipeline:
1. Load sample news articles
2. Analyze sentiment with `NewsImpactAnalyzer`
3. Visualize results with `sentiment_charts`
4. Run keyword analysis
5. Detect market events

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
from datetime import datetime, timezone, timedelta

from src.news_impact_analyzer import NewsImpactAnalyzer, NewsArticle
from src.keyword_analyzer import KeywordAnalyzer
from src.event_detector import EventDetector
import src.sentiment_charts as sc

%matplotlib inline

## 1. Sample News Articles

In [ ]:
articles = [
    NewsArticle(
        title='Apple beats earnings, record quarterly profit',
        content='Apple reported record-breaking earnings that beat all analyst expectations.',
        source='example', symbols=['AAPL'],
        published_at=datetime(2024, 1, 5, 21, 0, tzinfo=timezone.utc),
    ),
    NewsArticle(
        title='Tesla misses delivery targets, shares drop',
        content='Tesla delivered fewer vehicles than expected, causing a sharp decline in share price.',
        source='example', symbols=['TSLA'],
        published_at=datetime(2024, 1, 8, 14, 0, tzinfo=timezone.utc),
    ),
    NewsArticle(
        title='NVIDIA revenue surges on AI demand',
        content='NVIDIA reported massive revenue growth driven by AI chip demand from data centers.',
        source='example', symbols=['NVDA'],
        published_at=datetime(2024, 1, 12, 18, 0, tzinfo=timezone.utc),
    ),
    NewsArticle(
        title='Fed signals possible interest rate cut',
        content='Federal Reserve officials suggested a rate reduction may come in the next meeting.',
        source='example', symbols=['SPY', 'QQQ'],
        published_at=datetime(2024, 1, 15, 20, 0, tzinfo=timezone.utc),
    ),
    NewsArticle(
        title='Major pharmaceutical company faces FDA rejection',
        content='The FDA rejected the company\'s drug application, sending shares down 30%.',
        source='example', symbols=['PFE'],
        published_at=datetime(2024, 1, 18, 16, 0, tzinfo=timezone.utc),
    ),
    NewsArticle(
        title='Microsoft announces $10B AI investment partnership',
        content='Microsoft deepens its partnership with OpenAI with a massive new investment.',
        source='example', symbols=['MSFT'],
        published_at=datetime(2024, 1, 22, 9, 0, tzinfo=timezone.utc),
    ),
]
print(f'Loaded {len(articles)} articles')

## 2. Sentiment Analysis

In [ ]:
analyzer = NewsImpactAnalyzer(use_transformers=False)
impacts = analyzer.analyze_batch(articles)

print(f'{'Title':<50} {'Direction':<10} {'Score':>8} {'Conf':>6}')
print('-' * 78)
for imp in impacts:
    print(f'{imp.article.title[:48]:<50} {imp.direction:<10} {imp.impact_score:>+8.4f} {imp.confidence:>6.4f}')

## 3. Sentiment Over Time Chart

In [ ]:
timestamps = [imp.article.published_at for imp in impacts]
scores = [imp.impact_score for imp in impacts]

fig = sc.plot_sentiment_over_time(
    timestamps=timestamps,
    scores=scores,
    title='News Sentiment – January 2024',
)
fig

## 4. Sentiment by Symbol

In [ ]:
all_symbols = list({sym for imp in impacts for sym in imp.affected_symbols})
symbol_scores = {}
for sym in all_symbols:
    agg = analyzer.aggregate_impact(impacts, symbol=sym)
    if agg['articles_analyzed'] > 0:
        symbol_scores[sym] = agg['score']

fig = sc.plot_sentiment_by_symbol(symbol_scores=symbol_scores)
fig

## 5. Keyword Analysis

In [ ]:
kw_analyzer = KeywordAnalyzer(min_occurrences=1, top_n=10)
report = kw_analyzer.analyze(impacts)

print('TOP BULLISH KEYWORDS')
for word, score, count in report.top_bullish_keywords:
    print(f'  {word:<20} score={score:+.4f}  count={count}')

print()
print('TOP BEARISH KEYWORDS')
for word, score, count in report.top_bearish_keywords:
    print(f'  {word:<20} score={score:+.4f}  count={count}')

## 6. Event Detection

In [ ]:
detector = EventDetector(min_confidence=0.1)
events = detector.detect([imp.article for imp in impacts])

print(f'Detected {len(events)} events:')
for event in events:
    print(f'  {event}')

summary = detector.summarize(events)
print(f'\nEvent summary: {summary}')

## 7. Sentiment Distribution

In [ ]:
fig = sc.plot_sentiment_distribution(scores=scores)
fig